In [1]:
# Si estás en Colab y falta alguna librería, ejecuta esta celda.
# !pip install gymnasium opencv-python matplotlib numpy pandas

import gymnasium as gym
from gymnasium import spaces
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import json


# Entrenamiento String Art con agente mixto

Este notebook entrena usando **varias imágenes de entrada**. El agente mixto aprende una matriz `Q[clavo_origen, clavo_destino]` y la guarda para usarla después en otro notebook.

In [10]:
class EntornoHilograma(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"]}

    def __init__(self, ruta_imagen, num_clavos=120, resolucion=500, oscuridad_linea=20, max_lineas=1500, pixeles_linea=None):
        super().__init__()
        self.num_clavos = num_clavos
        self.res = resolucion
        self.centro = resolucion // 2
        self.radio = self.centro - 10
        self.oscuridad_linea = oscuridad_linea
        self.max_lineas = max_lineas
        self.ruta_imagen = ruta_imagen
        self.action_space = spaces.Discrete(self.num_clavos)
        self.observation_space = spaces.Box(low=0, high=255, shape=(self.res, self.res), dtype=np.int32)
        self.clavos = self._calcular_posiciones_clavos()
        self.pixeles_linea = pixeles_linea if pixeles_linea is not None else self._precalcular_pixeles_linea()
        self._preprocesar_imagen_objetivo()

    def _calcular_posiciones_clavos(self):
        angulos = np.linspace(0, 2 * np.pi, self.num_clavos, endpoint=False)
        x = self.centro + self.radio * np.cos(angulos)
        y = self.centro + self.radio * np.sin(angulos)
        return np.column_stack((x, y)).astype(np.int32)

    def _precalcular_pixeles_linea(self):
        print(f"[Entorno] Precalculando píxeles para {self.num_clavos * (self.num_clavos - 1) // 2} líneas...")
        pixeles_linea = {}
        n = self.num_clavos
        for i in range(n):
            for j in range(i + 1, n):
                mascara = np.zeros((self.res, self.res), dtype=np.uint8)
                cv2.line(mascara, tuple(self.clavos[i]), tuple(self.clavos[j]), 1, 1)
                ys, xs = np.nonzero(mascara)
                pixeles_linea[(i, j)] = (xs, ys)
                pixeles_linea[(j, i)] = (xs, ys)
        print("[Entorno] Precálculo terminado.")
        return pixeles_linea

    def _preprocesar_imagen_objetivo(self):
        img = cv2.imread(str(self.ruta_imagen), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"No se pudo cargar la imagen desde: {self.ruta_imagen}")
        h, w = img.shape
        dim_min = min(h, w)
        inicio_h = (h - dim_min) // 2
        inicio_w = (w - dim_min) // 2
        img_recortada = img[inicio_h:inicio_h + dim_min, inicio_w:inicio_w + dim_min]
        img_redimensionada = cv2.resize(img_recortada, (self.res, self.res))
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        img_clahe = clahe.apply(img_redimensionada)
        self.imagen_objetivo = 255 - img_clahe
        mascara = np.zeros((self.res, self.res), dtype=np.uint8)
        cv2.circle(mascara, (self.centro, self.centro), self.radio, 255, -1)
        self.imagen_objetivo = cv2.bitwise_and(self.imagen_objetivo, self.imagen_objetivo, mask=mascara)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.acumulador = np.zeros((self.res, self.res), dtype=np.int32)
        self.contador_lineas = 0
        self.clavo_actual = int(self.np_random.integers(0, self.num_clavos))
        return self.acumulador, {"clavo_actual": self.clavo_actual}

    def ganancia_linea(self, clavo_origen, clavo_destino):
        if clavo_origen == clavo_destino:
            return -100.0
        xs, ys = self.pixeles_linea[(clavo_origen, clavo_destino)]
        valores_objetivo = self.imagen_objetivo[ys, xs]
        valores_acumulador = self.acumulador[ys, xs]
        return float(np.sum(np.maximum(0, valores_objetivo - valores_acumulador)))

    def ganancias_desde_clavo_actual(self):
        ganancias = np.array([self.ganancia_linea(self.clavo_actual, j) for j in range(self.num_clavos)], dtype=float)
        ganancias[self.clavo_actual] = -np.inf
        return ganancias

    def step(self, action):
        siguiente_clavo = int(action)
        if siguiente_clavo == self.clavo_actual:
            info = {"clavo_actual": self.clavo_actual, "error": "Mismo clavo"}
            return self.acumulador, -100.0, False, False, info
        xs, ys = self.pixeles_linea[(self.clavo_actual, siguiente_clavo)]
        recompensa = self.ganancia_linea(self.clavo_actual, siguiente_clavo)
        self.acumulador[ys, xs] += self.oscuridad_linea
        self.clavo_actual = siguiente_clavo
        self.contador_lineas += 1
        truncated = self.contador_lineas >= self.max_lineas
        info = {"clavo_actual": self.clavo_actual, "linea_actual": self.contador_lineas}
        return self.acumulador, recompensa, False, truncated, info

    def render(self, mode="human"):
        lienzo = np.clip(255 - self.acumulador, 0, 255).astype(np.uint8)
        if mode == "rgb_array":
            return lienzo
        plt.figure(figsize=(6, 6))
        plt.imshow(lienzo, cmap="gray", vmin=0, vmax=255)
        plt.axis("off")
        plt.show()
        return lienzo


In [3]:
class AgenteMixtoQLearning:
    """Agente mixto: ganancia inmediata + novedad + salto + Q-table aprendida."""
    nombre = "Mixto con Q-table"

    def __init__(self, num_clavos, peso_ganancia=0.60, peso_novedad=0.15, peso_salto=0.10, peso_q=0.15,
                 alpha=0.12, gamma=0.80, epsilon=0.08, seed=42, q_table=None):
        self.num_clavos = num_clavos
        self.peso_ganancia = peso_ganancia
        self.peso_novedad = peso_novedad
        self.peso_salto = peso_salto
        self.peso_q = peso_q
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.rng = np.random.default_rng(seed)
        self.visitas = np.zeros(num_clavos, dtype=float)
        self.q_table = np.zeros((num_clavos, num_clavos), dtype=np.float32) if q_table is None else q_table.astype(np.float32)

    def _normalizar_q(self, origen):
        q = self.q_table[origen].astype(float).copy()
        q[origen] = -np.inf
        validas = np.isfinite(q)
        if not np.any(validas):
            return np.zeros(self.num_clavos)
        q_valid = q[validas]
        q_min, q_max = np.min(q_valid), np.max(q_valid)
        q_norm = np.zeros(self.num_clavos)
        if q_max > q_min:
            q_norm[validas] = (q_valid - q_min) / (q_max - q_min)
        return q_norm

    def seleccionar_accion(self, entorno):
        origen = entorno.clavo_actual
        ganancias = entorno.ganancias_desde_clavo_actual()
        validas = np.isfinite(ganancias)
        acciones_validas = np.where(validas)[0]
        if self.rng.random() < self.epsilon:
            return int(self.rng.choice(acciones_validas))

        g = np.zeros_like(ganancias, dtype=float)
        max_g = np.max(ganancias[validas]) if np.any(validas) else 1.0
        if max_g <= 0:
            max_g = 1.0
        g[validas] = ganancias[validas] / max_g
        novedad = 1.0 / (1.0 + self.visitas)
        dist_circular = np.array([min(abs(j - origen), self.num_clavos - abs(j - origen)) for j in range(self.num_clavos)], dtype=float)
        salto = dist_circular / (self.num_clavos / 2)
        q_norm = self._normalizar_q(origen)
        score = (self.peso_ganancia*g + self.peso_novedad*novedad + self.peso_salto*salto + self.peso_q*q_norm)
        score[~validas] = -np.inf
        return int(np.argmax(score))

    def actualizar(self, origen, accion, recompensa, nuevo_estado):
        r = recompensa / (abs(recompensa) + 10000.0)  # normalización estable 0..1 aprox.
        mejor_futuro = np.max(np.delete(self.q_table[nuevo_estado], nuevo_estado))
        td = r + self.gamma * mejor_futuro - self.q_table[origen, accion]
        self.q_table[origen, accion] += self.alpha * td
        self.visitas[accion] += 1


In [4]:
def entrenar_con_imagen(entorno, agente, seed=42, mostrar_cada=300):
    _, _ = entorno.reset(seed=seed)
    recompensa_total = 0.0
    terminado = truncado = False
    while not (terminado or truncado):
        origen = entorno.clavo_actual
        accion = agente.seleccionar_accion(entorno)
        _, recompensa, terminado, truncado, info = entorno.step(accion)
        agente.actualizar(origen, accion, recompensa, entorno.clavo_actual)
        recompensa_total += recompensa
        if info["linea_actual"] % mostrar_cada == 0:
            print(f"línea {info['linea_actual']} | recompensa acumulada {recompensa_total:.2f}")
    return recompensa_total, entorno.render(mode="rgb_array")


def entrenar_q_table(rutas_imagenes, configuracion, epocas=2, carpeta_salida="entrenamiento_mixto_qtable"):
    carpeta_salida = Path(carpeta_salida)
    carpeta_salida.mkdir(exist_ok=True)
    # Reutiliza píxeles de línea para acelerar cuando todas las imágenes usan la misma configuración.
    entorno_base = EntornoHilograma(rutas_imagenes[0], **configuracion)
    pixeles_linea = entorno_base.pixeles_linea
    agente = AgenteMixtoQLearning(num_clavos=configuracion["num_clavos"])
    registros = []
    for epoca in range(1, epocas + 1):
        for idx, ruta in enumerate(rutas_imagenes, 1):
            print("\n" + "="*70)
            print(f"Época {epoca}/{epocas} | Imagen {idx}/{len(rutas_imagenes)}: {ruta}")
            entorno = EntornoHilograma(ruta, pixeles_linea=pixeles_linea, **configuracion)
            recompensa, imagen_final = entrenar_con_imagen(entorno, agente, seed=epoca*100+idx,
                                                           mostrar_cada=max(1, configuracion['max_lineas']//5))
            salida_img = carpeta_salida / f"resultado_ep{epoca}_img{idx}.png"
            cv2.imwrite(str(salida_img), imagen_final)
            registros.append({"epoca": epoca, "imagen": str(ruta), "recompensa_total": recompensa, "resultado": str(salida_img)})
    q_path = carpeta_salida / "q_table_mixto.npz"
    np.savez_compressed(q_path,
        q_table=agente.q_table,
        num_clavos=configuracion["num_clavos"],
        resolucion=configuracion["resolucion"],
        oscuridad_linea=configuracion["oscuridad_linea"],
        max_lineas=configuracion["max_lineas"],
        configuracion=json.dumps(configuracion))
    pd.DataFrame(registros).to_csv(carpeta_salida / "historial_entrenamiento.csv", index=False)
    print(f"\nQ-table guardada en: {q_path}")
    return agente.q_table, pd.DataFrame(registros), q_path


## Configuración

Coloca varias imágenes en una carpeta o escribe sus rutas en la lista. Mientras más imágenes uses, mejor aprende patrones generales de líneas.

In [5]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:
# Opción A: carpeta con muchas imágenes
carpeta_imagenes = '/content/drive/MyDrive/SIS 420/SIS 420 ING/Proyecto/img'
extensiones = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp']
rutas_imagenes = []
for ext in extensiones:
    rutas_imagenes.extend(Path(carpeta_imagenes).glob(ext))
rutas_imagenes = sorted([str(p) for p in rutas_imagenes])

# Opción B: si prefieres, escribe las rutas manualmente:
# rutas_imagenes = [
#     '/content/drive/MyDrive/SIS 420/SIS 420 ING/Proyecto/gato.jpg',
#     '/content/drive/MyDrive/SIS 420/SIS 420 ING/Proyecto/perro.jpg',
# ]

print('Imágenes encontradas:', len(rutas_imagenes))
rutas_imagenes[:5]

Imágenes encontradas: 5


['/content/drive/MyDrive/SIS 420/SIS 420 ING/Proyecto/img/18b503289f4a5856bdf477cc137aee30_Original.jpg',
 '/content/drive/MyDrive/SIS 420/SIS 420 ING/Proyecto/img/689b6306c7bc49ba4c4be242_19-07-24-a-genetica-por-tras-das-cores-dos-gatos-blog.png',
 '/content/drive/MyDrive/SIS 420/SIS 420 ING/Proyecto/img/gato.jpg',
 '/content/drive/MyDrive/SIS 420/SIS 420 ING/Proyecto/img/img_como_limpiar_los_ojos_de_mi_gato_19137_600.jpg',
 '/content/drive/MyDrive/SIS 420/SIS 420 ING/Proyecto/img/survet-gato-caida-pelo-01.jpeg']

In [7]:
configuracion = {
    'num_clavos': 300,      # más clavos = mejor calidad, pero más lento
    'resolucion': 800,      # 400-600 rápido; 800 más fino
    'oscuridad_linea': 20,
    'max_lineas': 5000,     # 1000 rápido; 3000+ mejor resultado
}

epocas = 2  # número de vueltas sobre todas las imágenes

In [8]:
if len(rutas_imagenes) == 0:
    raise ValueError('No se encontraron imágenes. Revisa carpeta_imagenes o llena rutas_imagenes manualmente.')

q_table, historial, q_path = entrenar_q_table(rutas_imagenes, configuracion, epocas=epocas)
historial

[Entorno] Precalculando píxeles para 44850 líneas...
[Entorno] Precálculo terminado.

Época 1/2 | Imagen 1/5: /content/drive/MyDrive/SIS 420/SIS 420 ING/Proyecto/img/18b503289f4a5856bdf477cc137aee30_Original.jpg
línea 1000 | recompensa acumulada 94744164.00
línea 2000 | recompensa acumulada 166503304.00
línea 3000 | recompensa acumulada 219059369.00
línea 4000 | recompensa acumulada 255773529.00
línea 5000 | recompensa acumulada 280479215.00

Época 1/2 | Imagen 2/5: /content/drive/MyDrive/SIS 420/SIS 420 ING/Proyecto/img/689b6306c7bc49ba4c4be242_19-07-24-a-genetica-por-tras-das-cores-dos-gatos-blog.png
línea 1000 | recompensa acumulada 69903876.00
línea 2000 | recompensa acumulada 117062811.00
línea 3000 | recompensa acumulada 146404582.00
línea 4000 | recompensa acumulada 162973748.00
línea 5000 | recompensa acumulada 171497884.00

Época 1/2 | Imagen 3/5: /content/drive/MyDrive/SIS 420/SIS 420 ING/Proyecto/img/gato.jpg
línea 1000 | recompensa acumulada 72485636.00
línea 2000 | recompe

,epoca,imagen,recompensa_total,resultado
0,1,/content/drive/MyDrive/SIS 420/SIS 420 ING/Pro...,280479215.0,entrenamiento_mixto_qtable/resultado_ep1_img1.png
1,1,/content/drive/MyDrive/SIS 420/SIS 420 ING/Pro...,171497884.0,entrenamiento_mixto_qtable/resultado_ep1_img2.png
2,1,/content/drive/MyDrive/SIS 420/SIS 420 ING/Pro...,184344293.0,entrenamiento_mixto_qtable/resultado_ep1_img3.png
3,1,/content/drive/MyDrive/SIS 420/SIS 420 ING/Pro...,278136509.0,entrenamiento_mixto_qtable/resultado_ep1_img4.png
4,1,/content/drive/MyDrive/SIS 420/SIS 420 ING/Pro...,216591013.0,entrenamiento_mixto_qtable/resultado_ep1_img5.png
5,2,/content/drive/MyDrive/SIS 420/SIS 420 ING/Pro...,281732174.0,entrenamiento_mixto_qtable/resultado_ep2_img1.png
6,2,/content/drive/MyDrive/SIS 420/SIS 420 ING/Pro...,171520257.0,entrenamiento_mixto_qtable/resultado_ep2_img2.png
7,2,/content/drive/MyDrive/SIS 420/SIS 420 ING/Pro...,184126447.0,entrenamiento_mixto_qtable/resultado_ep2_img3.png
8,2,/content/drive/MyDrive/SIS 420/SIS 420 ING/Pro...,278338206.0,entrenamiento_mixto_qtable/resultado_ep2_img4.png
9,2,/content/drive/MyDrive/SIS 420/SIS 420 ING/Pro...,216420203.0,entrenamiento_mixto_qtable/resultado_ep2_img5.png


In [9]:
# Descargar Q-table en Google Colab
try:
    from google.colab import files
    files.download(str(q_path))
except Exception as e:
    print('Si no estás en Colab, descarga manualmente este archivo:', q_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Nota importante

La Q-table ayuda a reutilizar lo aprendido, pero el string art igual debe leer la imagen nueva y calcular ganancias de líneas. Por eso será **más rápido que volver a entrenar**, pero no necesariamente instantáneo. La calidad y velocidad dependen de `num_clavos`, `resolucion` y `max_lineas`.